# C01 — Convolutional Neural Networks from Scratch (NumPy)

> **Audience**: PhD students in ML/AI  ·  **Framework**: NumPy only  ·  **No library abstractions**

This notebook builds every component of a CNN by hand.
The goal is not speed — it is to make the mathematics completely transparent
before relying on framework abstractions.
Every tensor shape, gradient formula, and design decision is explicitly annotated.

**Contents**
1. Zero-padding
2. Single convolution step
3. Full convolution forward pass
4. Max- and average-pooling forward pass
5. Convolution backward pass — dA, dW, db
6. Pooling backward pass — max and average
7. Numerical gradient check
8. Complete forward–backward demonstration

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

np.random.seed(42)

# 1) The Convolution Operation

**Intuition**
A convolutional layer slides a small filter (kernel) across the input, computing an
element-wise multiply-then-sum at each position. Two key properties emerge:

- **Weight sharing** — the same filter is reused at every spatial position, giving far
  fewer parameters than a fully-connected layer.
- **Local connectivity** — each output value depends only on a small neighbourhood
  of the input (the *receptive field*).

**NHWC tensor convention (channels-last)**
All tensors use `(batch_num, height, width, channels)` order.
This matches NumPy's row-major layout and is the default in most frameworks.

**Output spatial dimensions**

Given input `(in_h × in_w)`, filter size `f`, padding `p`, stride `s`:

$$out\_h = \left\lfloor \frac{in\_h + 2p - f}{s} \right\rfloor + 1 \qquad
  out\_w = \left\lfloor \frac{in\_w + 2p - f}{s} \right\rfloor + 1$$

**Why care about this formula?**
Miscalculating output shapes is one of the most common causes of silent bugs
in novel architectures. Understanding it analytically prevents that.

## 1.1 Zero Padding

**Intuition**
Without padding, each convolution shrinks the spatial dimensions.
Zero-padding adds a border of zeros so the output preserves (or controls) the
spatial size.

**Two reasons to pad:**
1. **Same convolution** — preserve spatial resolution so deep networks do not
   collapse to 1×1.
2. **Edge information** — without padding, corner pixels contribute to only one
   output value; padding makes every pixel equally accessible.

**Sample I/O**
```
Input  X      : (batch_num=2, in_h=5, in_w=5, n_channels=3)
pad = 2
Output X_pad  : (2, 9, 9, 3)   ← each spatial dim grows by 2 × pad
```

In [ ]:
def zero_pad(X: np.ndarray, pad: int) -> np.ndarray:
    """
    Pads the spatial (H, W) axes of a 4-D input tensor with zeros.

    Only the height and width axes are padded; batch and channel axes
    are left unchanged. This is equivalent to adding a border of zeros
    around each image in the batch.

    Args:
        X   : Input images, shape (batch_num, in_h, in_w, n_channels)
        pad : Number of zero-rows/columns to add on each spatial border

    Returns:
        X_pad : Padded array, shape (batch_num, in_h + 2*pad, in_w + 2*pad, n_channels)
    """
    # Only pad spatial axes (axes 1 and 2); batch and channel axes have pad=0
    # (batch_num, in_h, in_w, n_channels) → (batch_num, in_h+2p, in_w+2p, n_channels)
    X_pad = np.pad(
        X,
        pad_width=((0, 0), (pad, pad), (pad, pad), (0, 0)),
        mode="constant",
        constant_values=0,
    )
    return X_pad


# ── Verification ──────────────────────────────────────────────────────────────
X = np.random.randn(4, 5, 5, 3)
X_pad = zero_pad(X, pad=2)

assert X_pad.shape == (4, 9, 9, 3), "Shape mismatch"
# Borders must be exactly zero
assert np.all(X_pad[:, :2, :, :] == 0), "Top border not zero"
assert np.all(X_pad[:, -2:, :, :] == 0), "Bottom border not zero"

print(f"X shape    : {X.shape}")      # (4, 5, 5, 3)
print(f"X_pad shape: {X_pad.shape}")  # (4, 9, 9, 3)

fig, axes = plt.subplots(1, 2, figsize=(8, 3))
for ax, data, title in zip(
    axes,
    [X[0, :, :, 0], X_pad[0, :, :, 0]],
    ["Original  (5×5)", "Padded  (9×9,  pad=2)"],
):
    ax.imshow(data, cmap="RdBu_r", vmin=-2, vmax=2)
    ax.set_title(title)
    ax.axis("off")
plt.tight_layout()
plt.show()

## 1.2 Single Convolution Step

**Intuition**
One convolution step applies one filter at one spatial position.
It is an element-wise multiply followed by a full summation — i.e. a dot product
between the flattened input slice and the flattened filter weights.

$$z = \sum_{h,w,c}\, a\_slice[h,w,c] \cdot W[h,w,c] + b$$

This is the indivisible unit. `conv_forward` in the next section
is just this operation repeated across all positions, all filters, all images.

**Sample I/O**
```
a_slice  : (f=3, f=3, n_channels_prev=4)   — receptive-field patch
W        : (3, 3, 4)                        — filter weights (same shape)
b        : (1, 1, 1)                        — bias scalar
→ z      : float scalar
```

In [ ]:
def conv_single_step(
    a_slice: np.ndarray, W: np.ndarray, b: np.ndarray
) -> float:
    """
    Applies one convolutional filter to one spatial position of the input.

    This is the atomic operation underlying the entire convolution:
    element-wise multiply then sum, plus a scalar bias.

    Args:
        a_slice : Receptive-field patch, shape (f, f, n_channels_prev)
        W       : Filter weights,        shape (f, f, n_channels_prev)
        b       : Bias,                  shape (1, 1, 1)

    Returns:
        z : Pre-activation scalar output at this spatial position
    """
    # Element-wise multiply then sum over all spatial and channel dimensions
    # (f, f, n_channels_prev) ⊙ (f, f, n_channels_prev) → scalar
    z = float(np.sum(a_slice * W) + float(b))
    return z


# ── Verification ──────────────────────────────────────────────────────────────
np.random.seed(1)
a_slice = np.random.randn(4, 4, 3)  # (f=4, f=4, n_channels_prev=3)
W_test  = np.random.randn(4, 4, 3)  # (f=4, f=4, n_channels_prev=3)
b_test  = np.random.randn(1, 1, 1)  # (1, 1, 1)

z = conv_single_step(a_slice, W_test, b_test)
print(f"z = {z:.6f}  (scalar as expected: {np.isscalar(z)})")

## 1.3 Convolution Forward Pass

**Intuition**
The full forward pass is `conv_single_step` repeated across four nested loops:

```
for each image   i  in [0 .. batch_num):
  for each row   h  in [0 .. out_h):
    for each col w  in [0 .. out_w):
      for each filter c in [0 .. n_filters):
          Z[i, h, w, c] = conv_single_step(slice, W[:,:,:,c], b[:,:,:,c])
```

The result `Z` is a feature map where each channel `c` records
where filter `c` pattern was activated across the spatial input.

**Sample I/O**
```
A_prev  : (batch_num=10, in_h=7, in_w=7, n_channels_prev=4)
W       : (f=3, f=3, n_channels_prev=4, n_filters=8)
b       : (1, 1, 1, n_filters=8)
stride=1, pad=1
→ Z    : (10, 7, 7, 8)    ← spatial preserved because pad=1, f=3, s=1
```

In [ ]:
def conv_forward(
    A_prev: np.ndarray,
    W: np.ndarray,
    b: np.ndarray,
    stride: int,
    pad: int,
) -> tuple:
    """
    Full forward pass of a single convolutional layer.

    Applies n_filters filters to every spatial position of every image in the
    batch, producing a 4-D output feature map Z.

    Args:
        A_prev : Input activations, shape (batch_num, in_h, in_w, n_channels_prev)
        W      : Filter weights,    shape (f, f, n_channels_prev, n_filters)
        b      : Biases,            shape (1, 1, 1, n_filters)
        stride : Convolution stride
        pad    : Zero-padding size

    Returns:
        Z     : Output feature map, shape (batch_num, out_h, out_w, n_filters)
        cache : (A_prev, W, b, stride, pad) — stored for the backward pass
    """
    batch_num, in_h, in_w, _ = A_prev.shape
    f, _, _, n_filters = W.shape

    # Compute output spatial dimensions using the standard CNN formula
    out_h = int((in_h + 2 * pad - f) / stride) + 1
    out_w = int((in_w + 2 * pad - f) / stride) + 1

    # Allocate output; filled in by the nested loops below
    # (batch_num, out_h, out_w, n_filters)
    Z = np.zeros((batch_num, out_h, out_w, n_filters))

    # Pad once before the loops — avoids redundant padding per sample
    # (batch_num, in_h, in_w, n_ch_prev) → (batch_num, in_h+2p, in_w+2p, n_ch_prev)
    A_prev_pad = zero_pad(A_prev, pad)

    for i in range(batch_num):
        a_prev_pad = A_prev_pad[i]  # (in_h+2p, in_w+2p, n_channels_prev)

        for h in range(out_h):
            vert_start = h * stride
            vert_end   = vert_start + f

            for w in range(out_w):
                horiz_start = w * stride
                horiz_end   = horiz_start + f

                for c in range(n_filters):
                    # Extract the receptive-field patch for this (h, w) position
                    # (f, f, n_channels_prev)
                    a_slice = a_prev_pad[vert_start:vert_end, horiz_start:horiz_end, :]

                    # Dot-product with filter c and store the scalar result
                    # scalar → Z[i, h, w, c]
                    Z[i, h, w, c] = conv_single_step(a_slice, W[:, :, :, c], b[:, :, :, c])

    cache = (A_prev, W, b, stride, pad)
    return Z, cache


# ── Verification ──────────────────────────────────────────────────────────────
np.random.seed(1)
A_prev = np.random.randn(10, 7, 7, 4)  # (batch_num, in_h, in_w, n_ch_prev)
W      = np.random.randn(3, 3, 4, 8)   # (f, f, n_ch_prev, n_filters)
b      = np.random.randn(1, 1, 1, 8)   # (1, 1, 1, n_filters)

Z, cache_conv = conv_forward(A_prev, W, b, stride=1, pad=1)

print(f"A_prev : {A_prev.shape}")   # (10, 7, 7, 4)
print(f"W      : {W.shape}")        # (3, 3, 4, 8)
print(f"Z      : {Z.shape}")        # (10, 7, 7, 8)  ← spatial preserved
print(f"Z mean : {Z.mean():.4f}")
print(f"Z[3,2,1] = {np.round(Z[3, 2, 1, :], 3)}")

# 2) Pooling

**Intuition**
Pooling reduces spatial resolution, providing:

- **Translation invariance** — small shifts in input produce identical output.
- **Reduced computation** — fewer activations to process in subsequent layers.
- **Larger effective receptive field** — downstream neurons see a wider input region.

Pooling has **no learnable parameters** — it is a fixed aggregation operation.

| Mode    | Operation | Typical use case |
|---------|-----------|-----------------|
| Max     | max over window | Feature detection (is the pattern present?) |
| Average | mean over window | Smooth feature maps, global average pooling at the end |

**Sample I/O**
```
A_prev : (batch_num=2, in_h=8, in_w=8, n_channels=3)
f=2, stride=2
→ A   : (2, 4, 4, 3)   ← spatial halved (non-overlapping windows, no padding)
```

**Output size (no padding for pooling)**
$$out\_h = \left\lfloor \frac{in\_h - f}{s} \right\rfloor + 1$$

In [ ]:
def pool_forward(
    A_prev: np.ndarray, f: int, stride: int, mode: str = "max"
) -> tuple:
    """
    Forward pass of a pooling layer (max or average).

    Channels never interact during pooling — each channel is pooled
    independently. This is why pooling has no parameters.

    Design decision — why pool per-channel independently?
    Mixing channels during pooling would destroy the feature-map interpretation
    that each channel encodes one filter's response.

    Args:
        A_prev : Input activations, shape (batch_num, in_h, in_w, n_channels)
        f      : Pooling window size (f × f square)
        stride : Sliding step of the window
        mode   : 'max' for max pooling, 'average' for average pooling

    Returns:
        A     : Pooled output, shape (batch_num, out_h, out_w, n_channels)
        cache : (A_prev, f, stride, mode) — stored for the backward pass
    """
    batch_num, in_h, in_w, n_channels = A_prev.shape

    out_h = int((in_h - f) / stride) + 1
    out_w = int((in_w - f) / stride) + 1

    # (batch_num, out_h, out_w, n_channels)
    A = np.zeros((batch_num, out_h, out_w, n_channels))

    for i in range(batch_num):
        for h in range(out_h):
            vert_start = h * stride
            vert_end   = vert_start + f

            for w in range(out_w):
                horiz_start = w * stride
                horiz_end   = horiz_start + f

                # Slice all channels simultaneously for this window
                # (f, f, n_channels)
                a_window = A_prev[i, vert_start:vert_end, horiz_start:horiz_end, :]

                if mode == "max":
                    # Reduce spatial dims to scalar per channel via max
                    # (f, f, n_channels) → (n_channels,)
                    A[i, h, w, :] = np.max(a_window, axis=(0, 1))
                elif mode == "average":
                    # (f, f, n_channels) → (n_channels,)
                    A[i, h, w, :] = np.mean(a_window, axis=(0, 1))
                else:
                    raise ValueError(f"mode must be 'max' or 'average', got '{mode}'")

    cache = (A_prev, f, stride, mode)
    return A, cache


# ── Verification ──────────────────────────────────────────────────────────────
np.random.seed(1)
A_prev = np.random.randn(2, 8, 8, 3)  # (batch_num, 8×8, 3 channels)

A_max, _ = pool_forward(A_prev, f=2, stride=2, mode="max")
A_avg, _ = pool_forward(A_prev, f=2, stride=2, mode="average")

print(f"Input shape  : {A_prev.shape}")  # (2, 8, 8, 3)
print(f"Max pool out : {A_max.shape}")   # (2, 4, 4, 3)
print(f"Avg pool out : {A_avg.shape}")   # (2, 4, 4, 3)

fig, axes = plt.subplots(1, 3, figsize=(12, 3))
for ax, data, title in zip(
    axes,
    [A_prev[0, :, :, 0], A_max[0, :, :, 0], A_avg[0, :, :, 0]],
    ["Input (8×8)", "Max pool (4×4)", "Avg pool (4×4)"],
):
    ax.imshow(data, cmap="RdBu_r")
    ax.set_title(title)
    ax.axis("off")
plt.tight_layout()
plt.show()

# 3) Backpropagation

**Why implement this by hand?**
Modern frameworks compute gradients automatically. But understanding the
backward pass answers a fundamental research question:
*exactly where does each unit of gradient flow, and why?*

This reveals the source of vanishing gradients, dead neurons, and
filter-learning dynamics — all concepts that appear repeatedly in the literature.

**Chain rule for the convolution**

Let $L$ be the scalar loss. The upstream gradient $dZ$ has shape `(batch_num, out_h, out_w, n_filters)`.

$$\frac{\partial L}{\partial A_{prev}[i, h\!:\!h+f,\, w\!:\!w+f,\, :]} \mathrel{+}= W[:,:,:,c] \cdot dZ[i,h,w,c]$$

$$\frac{\partial L}{\partial W[:,:,:,c]} \mathrel{+}= a\_slice \cdot dZ[i,h,w,c]$$

$$\frac{\partial L}{\partial b[:,:,:,c]} \mathrel{+}= dZ[i,h,w,c]$$

**Why `+=` (accumulation)?**
Each filter weight $W[:,:,:,c]$ is *shared* across all `out_h × out_w` positions.
Its total gradient is the *sum* of contributions from every output position
where it participated — this is the chain rule applied to a shared parameter.

## 3.1 Convolution Backward Pass

**Key structural insight**
The backward pass mirrors the forward pass exactly — same four nested loops,
same slice indices. The only difference:

- **Forward**: read `W`, write `Z`
- **Backward**: read `dZ`, accumulate into `dA_prev`, `dW`, `db`

In [ ]:
def conv_backward(dZ: np.ndarray, cache: tuple) -> tuple:
    """
    Backward pass of a convolutional layer.

    Computes dA_prev, dW, and db from the upstream gradient dZ using the
    same spatial loop structure as conv_forward.

    Args:
        dZ    : Upstream gradient, shape (batch_num, out_h, out_w, n_filters)
        cache : (A_prev, W, b, stride, pad) saved during the forward pass

    Returns:
        dA_prev : Gradient w.r.t. input,   shape (batch_num, in_h, in_w, n_channels_prev)
        dW      : Gradient w.r.t. weights, shape (f, f, n_channels_prev, n_filters)
        db      : Gradient w.r.t. bias,    shape (1, 1, 1, n_filters)
    """
    A_prev, W, b, stride, pad = cache
    batch_num, in_h, in_w, _ = A_prev.shape
    f, _, _, n_filters        = W.shape
    _, out_h, out_w, _        = dZ.shape

    # Initialise gradients to zero before accumulation
    dA_prev = np.zeros_like(A_prev)  # (batch_num, in_h, in_w, n_channels_prev)
    dW      = np.zeros_like(W)       # (f, f, n_channels_prev, n_filters)
    db      = np.zeros_like(b)       # (1, 1, 1, n_filters)

    # Pad A_prev and dA_prev identically so slice indices match the forward pass
    # (batch_num, in_h+2p, in_w+2p, n_ch_prev)
    A_prev_pad   = zero_pad(A_prev, pad)
    dA_prev_pad  = zero_pad(dA_prev, pad)

    for i in range(batch_num):
        for h in range(out_h):
            vert_start = h * stride
            vert_end   = vert_start + f

            for w in range(out_w):
                horiz_start = w * stride
                horiz_end   = horiz_start + f

                for c in range(n_filters):
                    # Retrieve the same slice that was used in the forward pass
                    # (f, f, n_channels_prev)
                    a_slice = A_prev_pad[i, vert_start:vert_end, horiz_start:horiz_end, :]
                    dz      = dZ[i, h, w, c]  # scalar upstream gradient

                    # Propagate gradient back to the input patch
                    # (f, f, n_ch_prev) += (f, f, n_ch_prev) * scalar
                    dA_prev_pad[i, vert_start:vert_end, horiz_start:horiz_end, :] += (
                        W[:, :, :, c] * dz
                    )

                    # Accumulate filter gradient — each position contributes once per filter
                    # (f, f, n_ch_prev) += (f, f, n_ch_prev) * scalar
                    dW[:, :, :, c] += a_slice * dz

                    # Accumulate bias gradient — one scalar per filter per position
                    db[:, :, :, c] += dz

    # Remove padding to recover original spatial dimensions
    # (batch_num, in_h+2p, in_w+2p, n_ch) → (batch_num, in_h, in_w, n_ch)
    dA_prev = dA_prev_pad[:, pad:-pad, pad:-pad, :] if pad > 0 else dA_prev_pad

    return dA_prev, dW, db


# ── Verification ──────────────────────────────────────────────────────────────
np.random.seed(1)
A_prev_v = np.random.randn(10, 4, 4, 3)
W_v      = np.random.randn(2, 2, 3, 8)
b_v      = np.random.randn(1, 1, 1, 8)

Z_v, cache_v = conv_forward(A_prev_v, W_v, b_v, stride=2, pad=2)
dZ_v         = np.random.randn(*Z_v.shape)

dA_prev_v, dW_v, db_v = conv_backward(dZ_v, cache_v)

print(f"Z      : {Z_v.shape}")         # (10, 4, 4, 8)
print(f"dA_prev: {dA_prev_v.shape}")   # (10, 4, 4, 3)  ← matches A_prev
print(f"dW     : {dW_v.shape}")        # (2, 2, 3, 8)   ← matches W
print(f"db     : {db_v.shape}")        # (1, 1, 1, 8)   ← matches b

## 3.2 Pooling Backward Pass

**Max pooling backward — sparse gradient**
Only the position that held the maximum value influences the output, so only
it receives gradient. All other positions get zero.
This sparsity is a *feature*, not a bug: it forces the preceding layers
to produce activations that are strongly distinguishable from neighbours.

**Average pooling backward — uniform gradient**
Every position contributed equally to the output (they were averaged),
so the gradient is distributed uniformly.

**Research implication**
Max pooling creates sparse gradient signals that can effectively silence
neurons that aren't the local maximum. If most neurons in a feature map
are consistently not the maximum, they may effectively stop learning.

In [ ]:
def create_mask_from_window(x: np.ndarray) -> np.ndarray:
    """
    Builds a binary mask identifying the maximum position in a 2-D pooling window.

    During max-pool backprop, gradient flows only through the maximum position.
    If multiple positions share the same max value, they split the gradient equally.

    Args:
        x    : 2-D pooling window, shape (f, f)

    Returns:
        mask : Boolean array, shape (f, f); True where x == max(x)
    """
    # (f, f) → (f, f) bool; True only at the maximum position(s)
    return x == np.max(x)


def distribute_value(dz: float, shape: tuple) -> np.ndarray:
    """
    Distributes a scalar gradient uniformly across an (f, f) window.

    Used in average-pooling backward: since every element contributed 1/(f*f)
    to the forward output, every element receives dz / (f*f) of the gradient.

    Args:
        dz    : Upstream gradient (scalar)
        shape : Target shape (f_h, f_w)

    Returns:
        a : Uniform gradient array, shape (f_h, f_w), each element = dz / (f_h * f_w)
    """
    f_h, f_w = shape
    # scalar → (f_h, f_w); every element receives an equal share
    return np.full(shape, dz / (f_h * f_w))


def pool_backward(dA: np.ndarray, cache: tuple) -> np.ndarray:
    """
    Backward pass of a pooling layer (max or average).

    Args:
        dA    : Upstream gradient, shape (batch_num, out_h, out_w, n_channels)
        cache : (A_prev, f, stride, mode) saved during the forward pass

    Returns:
        dA_prev : Gradient w.r.t. input, shape (batch_num, in_h, in_w, n_channels)
    """
    A_prev, f, stride, mode = cache
    batch_num, in_h, in_w, n_channels = A_prev.shape
    _, out_h, out_w, _                = dA.shape

    # (batch_num, in_h, in_w, n_channels)
    dA_prev = np.zeros_like(A_prev)

    for i in range(batch_num):
        for h in range(out_h):
            vert_start = h * stride
            vert_end   = vert_start + f

            for w in range(out_w):
                horiz_start = w * stride
                horiz_end   = horiz_start + f

                for c in range(n_channels):
                    if mode == "max":
                        a_window = A_prev[i, vert_start:vert_end, horiz_start:horiz_end, c]

                        # Binary mask: gradient flows only through the maximum position
                        # (f, f) → (f, f) bool
                        mask = create_mask_from_window(a_window)

                        # Multiply scalar upstream gradient by the mask
                        # scalar * (f, f) bool → (f, f)
                        dA_prev[i, vert_start:vert_end, horiz_start:horiz_end, c] += (
                            mask * dA[i, h, w, c]
                        )

                    elif mode == "average":
                        # Distribute gradient uniformly — all positions contributed equally
                        # scalar → (f, f)
                        dA_prev[i, vert_start:vert_end, horiz_start:horiz_end, c] += (
                            distribute_value(dA[i, h, w, c], (f, f))
                        )

    return dA_prev


# ── Verification ──────────────────────────────────────────────────────────────
np.random.seed(1)
A_pool = np.random.randn(5, 5, 3, 2)

A_max_p, cache_max = pool_forward(A_pool, f=2, stride=1, mode="max")
A_avg_p, cache_avg = pool_forward(A_pool, f=2, stride=1, mode="average")

dA_prev_max = pool_backward(np.random.randn(*A_max_p.shape), cache_max)
dA_prev_avg = pool_backward(np.random.randn(*A_avg_p.shape), cache_avg)

print(f"Max pool dA_prev shape : {dA_prev_max.shape}")  # (5, 5, 3, 2)
print(f"Avg pool dA_prev shape : {dA_prev_avg.shape}")  # (5, 5, 3, 2)

# Average pool conserves gradient mass: sum of dA_prev == sum of dA
dA_avg_test = np.ones_like(A_avg_p)  # uniform upstream gradient
dA_prev_avg_test = pool_backward(dA_avg_test, cache_avg)
mass_in  = dA_avg_test.sum()
mass_out = dA_prev_avg_test.sum()
print(f"Avg pool gradient mass conserved: {np.isclose(mass_in, mass_out)}  "
      f"(in={mass_in:.4f}, out={mass_out:.4f})")

# 4) Numerical Gradient Check

**The gold standard for verifying backward passes**

The finite-difference approximation estimates gradients numerically, completely
independently of your analytical derivation. Comparing the two is the most
reliable way to catch bugs in backward pass implementations.

**Central difference formula** (more accurate than forward difference)

$$\frac{\partial f}{\partial x_i} \approx \frac{f(x + \epsilon\, e_i) - f(x - \epsilon\, e_i)}{2\epsilon}$$

where $e_i$ is the unit vector in dimension $i$ and $\epsilon \approx 10^{-5}$.

**Relative error threshold**

$$\text{relative error} = \frac{\|\nabla_{\text{analytic}} - \nabla_{\text{numeric}}\|_2}
{\|\nabla_{\text{analytic}}\|_2 + \|\nabla_{\text{numeric}}\|_2} $$

| Value | Interpretation |
|-------|---------------|
| $< 10^{-7}$ | Numerical precision noise; effectively zero |
| $< 10^{-5}$ | **Correct implementation** |
| $10^{-3}$   | Probably a bug |
| $> 10^{-1}$ | Definitely a bug |

In [ ]:
def numerical_gradient(func, param: np.ndarray, eps: float = 1e-5) -> np.ndarray:
    """
    Estimates gradients using central finite differences.

    Perturbs each element of `param` by +eps and -eps independently,
    measures the change in the scalar output of `func`, and computes the slope.

    Note: O(2 * param.size) function evaluations — use only for small tensors.

    Args:
        func  : Function mapping param → scalar
        param : Parameter array to differentiate; will be modified in-place
                and restored after each evaluation
        eps   : Perturbation magnitude (default 1e-5)

    Returns:
        grad_num : Estimated gradient, same shape as param
    """
    grad_num = np.zeros_like(param)
    it = np.nditer(param, flags=["multi_index"], op_flags=["readwrite"])

    while not it.finished:
        idx = it.multi_index
        x_orig = param[idx]

        param[idx] = x_orig + eps
        f_plus  = func(param)

        param[idx] = x_orig - eps
        f_minus = func(param)

        param[idx] = x_orig  # restore

        # Central-difference approximation — O(eps^2) error vs O(eps) for forward diff
        grad_num[idx] = (f_plus - f_minus) / (2.0 * eps)
        it.iternext()

    return grad_num


def relative_error(g_a: np.ndarray, g_n: np.ndarray) -> float:
    """
    Computes the relative L2 error between analytic and numerical gradients.

    Args:
        g_a : Analytic gradient from backward pass
        g_n : Numerical gradient from finite differences

    Returns:
        Relative error scalar; < 1e-5 indicates a correct implementation
    """
    numerator   = np.linalg.norm(g_a - g_n)
    denominator = np.linalg.norm(g_a) + np.linalg.norm(g_n) + 1e-12
    return float(numerator / denominator)


# ── Run checks on dW and dA_prev ──────────────────────────────────────────────
np.random.seed(0)
A_chk  = np.random.randn(2, 5, 5, 3)   # small tensors — grad check is O(2 * size)
W_chk  = np.random.randn(3, 3, 3, 4)
b_chk  = np.random.randn(1, 1, 1, 4)
s_chk, p_chk = 1, 1

# Use sum(Z) as a proxy loss so that dL/dZ = ones_like(Z)
loss_fn = lambda x, w, b_: float(np.sum(conv_forward(x, w, b_, s_chk, p_chk)[0]))

Z_chk, cache_chk = conv_forward(A_chk, W_chk, b_chk, s_chk, p_chk)
dZ_ones = np.ones_like(Z_chk)  # dL/dZ = 1 for loss = sum(Z)

dA_analytic, dW_analytic, _ = conv_backward(dZ_ones, cache_chk)

# Numerically estimate dW
dW_numeric = numerical_gradient(
    lambda w: loss_fn(A_chk, w, b_chk), W_chk.copy()
)

# Numerically estimate dA_prev
dA_numeric = numerical_gradient(
    lambda a: loss_fn(a, W_chk, b_chk), A_chk.copy()
)

err_W = relative_error(dW_analytic, dW_numeric)
err_A = relative_error(dA_analytic, dA_numeric)

print(f"dW relative error : {err_W:.2e}   {'PASS' if err_W < 1e-5 else 'FAIL'}")
print(f"dA relative error : {err_A:.2e}   {'PASS' if err_A < 1e-5 else 'FAIL'}")

# 5) Complete Forward–Backward Demonstration

**Assembling a minimal two-layer network**
```
Input → Conv(f=3, n_filters=4, s=1, p=1) → ReLU → MaxPool(f=2, s=2) → MSE loss
```
One full forward pass followed by one full backward pass,
printing all intermediate shapes.

This is exactly what `torch.nn.Conv2d` + `torch.nn.MaxPool2d` execute internally
when you call `loss.backward()`. Understanding this loop is the foundation
for debugging gradient flow in any architecture.

In [ ]:
np.random.seed(7)

# Synthetic input: 4 images, 8×8, 1 channel
# (batch_num=4, in_h=8, in_w=8, n_channels=1)
X_demo = np.random.randn(4, 8, 8, 1)

# Convolution weights (small init for stable forward pass)
W1 = np.random.randn(3, 3, 1, 4) * 0.1  # (f=3, f=3, n_ch_prev=1, n_filters=4)
b1 = np.zeros((1, 1, 1, 4))              # (1, 1, 1, n_filters=4)

# ─── FORWARD PASS ─────────────────────────────────────────────────────────────

# Conv layer: (4, 8, 8, 1) → (4, 8, 8, 4)   [stride=1, pad=1 preserves spatial]
Z1, cache1 = conv_forward(X_demo, W1, b1, stride=1, pad=1)

# ReLU activation: zero out negative pre-activations
# (4, 8, 8, 4) → (4, 8, 8, 4)   [same shape, in-place threshold at 0]
A1 = np.maximum(Z1, 0)

# Max pool: halve spatial dimensions
# (4, 8, 8, 4) → (4, 4, 4, 4)
A2, cache2 = pool_forward(A1, f=2, stride=2, mode="max")

# Toy loss: mean squared error from an all-zero target
# scalar
loss = float(np.mean(A2 ** 2))

# ─── BACKWARD PASS ────────────────────────────────────────────────────────────

# dL/dA2 = 2 * A2 / A2.size   (derivative of mean(A2^2) w.r.t. A2)
# (4, 4, 4, 4)
dA2 = 2.0 * A2 / A2.size

# Backprop through max pool
# (4, 4, 4, 4) → (4, 8, 8, 4)
dA1 = pool_backward(dA2, cache2)

# Backprop through ReLU: gate gradient — zero where Z1 <= 0 (dead units)
# (4, 8, 8, 4)
dZ1 = dA1 * (Z1 > 0)

# Backprop through conv layer
# (4, 8, 8, 4) → dX:(4,8,8,1)  dW1:(3,3,1,4)  db1:(1,1,1,4)
dX, dW1_grad, db1_grad = conv_backward(dZ1, cache1)

# ─── SHAPE SUMMARY ────────────────────────────────────────────────────────────
print("FORWARD")
print(f"  Input X       : {X_demo.shape}")   # (4, 8, 8, 1)
print(f"  After Conv1   : {Z1.shape}")        # (4, 8, 8, 4)
print(f"  After ReLU    : {A1.shape}")        # (4, 8, 8, 4)
print(f"  After MaxPool : {A2.shape}")        # (4, 4, 4, 4)
print(f"  Loss          : {loss:.6f}")

print("\nBACKWARD")
print(f"  dA2    : {dA2.shape}")              # (4, 4, 4, 4)
print(f"  dA1    : {dA1.shape}")              # (4, 8, 8, 4)
print(f"  dZ1    : {dZ1.shape}")              # (4, 8, 8, 4)
print(f"  dX     : {dX.shape}")               # (4, 8, 8, 1) ← matches input
print(f"  dW1    : {dW1_grad.shape}")         # (3, 3, 1, 4) ← matches W1
print(f"  db1    : {db1_grad.shape}")         # (1, 1, 1, 4) ← matches b1

print("\nGRADIENT NORMS  (non-zero means gradients are flowing)")
print(f"  ||dX ||  = {np.linalg.norm(dX):.5f}")
print(f"  ||dW1||  = {np.linalg.norm(dW1_grad):.5f}")
print(f"  ||db1||  = {np.linalg.norm(db1_grad):.5f}")

# Summary

| Component | Forward | Backward |
|---|---|---|
| `zero_pad` | Add border of zeros | No gradient (non-parametric) |
| `conv_single_step` | Element-wise dot product + bias | Feeds into `conv_backward` |
| `conv_forward` | Slide filters over all positions | `conv_backward` mirrors the same loops |
| `pool_forward` | Max or mean reduce per window | Sparse (max) or uniform (avg) gradient |

**Key takeaways for research**

- **Shape arithmetic**: always derive output dimensions analytically
  before training; miscalculated shapes produce silent bugs.
- **Gradient sparsity**: max-pool creates sparse gradients;
  average-pool distributes them uniformly — this choice affects feature-map learning dynamics.
- **Weight sharing**: each filter's gradient is the *sum* over all positions
  where it contributed. This is why CNNs converge faster than fully-connected
  networks on structured inputs.
- **Framework equivalence**: `torch.nn.Conv2d(in_channels=1, out_channels=4,
  kernel_size=3, padding=1)` executes the exact `conv_forward` computation
  implemented here, with automatic gradient via `conv_backward`.